In [ ]:
import os
os.chdir("..")

from data.data import get_lp_dataloaders
from utils.utils import set_seed
from scripts.lp_script import construct_backbone, train_classifier, test
import torch
from models.models import Classifier

SEED = 2952
set_seed(SEED, deterministic=True, benchmark=False) # benchmark=False for reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
######## HYPERPARAMETERS ########
batch_size = 512
workers = 32
epochs=100
lr=3e-4
wd=1e-4
out_dir="runs/exp1"
arch = 'vit_s'
ssl_model_path = './checkpoints_vit_s/checkpoint_0300_v2.pth.tar'
use_amp=True

In [ ]:
######## PREPARE DATA, MODEL ########
train_loader, val_loader, test_loader = get_lp_dataloaders(batch_size=batch_size, workers=workers)
backbone = construct_backbone(arch=arch, ckpt_path=ssl_model_path, device=device)
model = Classifier(backbone=backbone, num_classes=10)

In [ ]:
######## TRAINING ########
train_classifier(epochs=epochs,
                 model=model,
                 train_loader=train_loader,
                 val_loader=val_loader,
                 lr=lr,
                 wd=wd,
                 device=device,
                 out_dir=out_dir,
                 use_amp=use_amp,
                 print_freq=10)

In [ ]:
######## TESTING ########
best_model_path = f"{out_dir}/best.ckpt"
test(model, test_loader, device, ckpt_path=best_model_path)